# 고정 RAG와 Agentic RAG 비교

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.


## 패키지 설치


In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3 pillow

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "labs/part6/ch21-create-agent/pyproject.toml").is_file())
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("langchain", version("langchain"))
print("langgraph", version("langgraph"))
print("API 키 존재 여부 확인 완료")
%pip install -q numpy

langchain 1.4.0
langgraph 1.2.11
API 키 존재 여부 확인 완료


Note: you may need to restart the kernel to use updated packages.

## Ch14 문서와 평가셋 읽기

공정한 비교에서는 두 경로가 같은 문서, 질문, 정답 기준, 검색 함수를 사용해야 합니다. `cases`는 Ch14 평가 질문 중 세 건을 고정하고, `document_vectors`는 두 경로가 공유하는 임베딩 색인입니다. `retrieve(query, top_k=2)`는 질문 벡터와 모든 문서 벡터의 코사인 유사도를 계산해 상위 두 문서와 점수를 반환합니다. 여기서 반환되는 점수는 답의 정확도가 아니라 질문과 문서 벡터의 가까움을 나타냅니다.

고정 경로는 애플리케이션이 `retrieve`를 호출한 뒤 원문을 모델 프롬프트에 넣습니다. 에이전트 경로는 같은 `retrieve`를 감싼 `search_documents`를 모델이 선택합니다. `usage_total`은 한 실행의 모든 `AIMessage.usage_metadata`를 합쳐 모델 호출 수와 토큰을 세며, `search_events`는 도구 호출 횟수와 검색어를 별도로 남깁니다. 서로 다른 측정값을 같은 비용 또는 품질 지표로 합치지 않습니다.


In [3]:
# Ch14 원본 documents와 queries에서 비교할 C01·C05·C07을 그대로 선택합니다.
import json
from pathlib import Path
data_dir = ROOT / "labs/part4/ch14-rag-patterns/data"
documents = json.loads((data_dir / "documents.json").read_text(encoding="utf-8"))
queries = json.loads((data_dir / "queries.json").read_text(encoding="utf-8"))
cases = [queries[index] for index in [0, 4, 6]]
print("원천:", data_dir.relative_to(ROOT))
print("문서 수:", len(documents), "평가 질문:", [case["qid"] for case in cases])
assert len(documents) == 12 and len(cases) == 3

원천: labs/part4/ch14-rag-patterns/data
문서 수: 12 평가 질문: ['C01', 'C05', 'C07']


## 공통 임베딩 색인 만들기


In [4]:
# 열두 문서의 임베딩을 단위 벡터로 정규화해 두 RAG 경로가 공유할 색인을 만듭니다.
import numpy as np
from langchain_openai import OpenAIEmbeddings
embedder = OpenAIEmbeddings(model="text-embedding-3-small")
document_vectors = np.array(embedder.embed_documents([item["text"] for item in documents]))
document_vectors /= np.linalg.norm(document_vectors, axis=1, keepdims=True)
print("색인 모양:", document_vectors.shape)
assert document_vectors.shape[0] == len(documents)

색인 모양: (12, 1536)


## 공통 검색 함수 정의


In [5]:
# 질문 벡터와 문서 벡터의 코사인 유사도로 상위 두 원문과 점수를 반환합니다.
def retrieve(query: str, top_k: int = 2) -> list[dict]:
    """질문 임베딩과 가까운 원문을 top_k개 반환합니다."""
    query_vector = np.array(embedder.embed_query(query))
    query_vector /= np.linalg.norm(query_vector)
    scores = document_vectors @ query_vector
    indices = np.argsort(scores)[::-1][:top_k]
    return [{**documents[index], "score": round(float(scores[index]), 4)} for index in indices]

sample_hits = retrieve(cases[0]["question"])
print([(item["doc_id"], item["score"]) for item in sample_hits])
assert cases[0]["gold_doc_id"] in [item["doc_id"] for item in sample_hits]

[('LIB-004', 0.5752), ('LIB-001', 0.4724)]


## 고정 RAG와 Agentic RAG 준비


In [6]:
# 같은 모델과 색인을 쓰되 고정 경로는 직접 검색하고 Agent 경로는 search_documents 도구를 선택합니다.
from time import perf_counter
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none")
search_events: list[dict] = []
@tool
def search_documents(query: str) -> str:
    """Ch14 교육센터 문서에서 질문과 가까운 원문 두 건을 검색합니다."""
    hits = retrieve(query)
    search_events.append({"query": query, "doc_ids": [item["doc_id"] for item in hits]})
    return "\n".join(f"[{item['doc_id']}] {item['text']}" for item in hits)
agent = create_agent(model, [search_documents], system_prompt="규정 질문에는 search_documents를 사용하세요. 검색 원문에 없는 내용은 만들지 마세요.")
print("비교 경로 준비 완료")

비교 경로 준비 완료


## 호출별 사용량 집계 함수


In [7]:
# 한 실행에 포함된 모든 AIMessage의 입력·출력 토큰과 모델 호출 수를 합산합니다.
def usage_total(messages: list) -> dict[str, int]:
    """모든 AIMessage의 입력·출력 토큰을 합산합니다."""
    usages = [message.usage_metadata for message in messages if getattr(message, "usage_metadata", None)]
    return {"input_tokens": sum(item["input_tokens"] for item in usages), "output_tokens": sum(item["output_tokens"] for item in usages), "model_calls": len(usages)}

assert usage_total([]) == {"input_tokens": 0, "output_tokens": 0, "model_calls": 0}

## 같은 평가셋으로 두 경로 실행


In [8]:
# 각 질문의 원문, 모델 답, 검색 trace, 전체 지연을 두 경로에서 빠짐없이 보존합니다.
records: list[dict] = []
for case in cases:
    hits = retrieve(case["question"])
    context = "\n".join(f"[{item['doc_id']}] {item['text']}" for item in hits)
    started = perf_counter()
    fixed = model.invoke(f"다음 원문만 사용해 답하세요.\n{context}\n질문: {case['question']}")
    fixed_latency = perf_counter() - started
    search_events.clear()
    started = perf_counter()
    agent_result = agent.invoke({"messages": [{"role": "user", "content": case["question"]}]})
    agent_latency = perf_counter() - started
    records.append({"case": case, "fixed_hits": hits, "fixed": fixed, "fixed_latency": fixed_latency, "agent_result": agent_result, "agent_latency": agent_latency, "search_events": list(search_events)})
print("실행 행:", len(records))
assert len(records) == 3

실행 행: 3


## 원문 답변과 실행 지표 출력


In [9]:
# 원문 답변과 문서 id, 모델·도구 호출 수, 토큰, 지연을 질문별 한 행으로 출력합니다.
for record in records:
    fixed_usage = usage_total([record["fixed"]])
    agent_messages = record["agent_result"]["messages"]
    agent_usage = usage_total(agent_messages)
    tool_calls = sum(len(getattr(message, "tool_calls", [])) for message in agent_messages)
    print({"qid": record["case"]["qid"], "fixed_answer": record["fixed"].text, "agent_answer": agent_messages[-1].text, "fixed_doc_ids": [item["doc_id"] for item in record["fixed_hits"]], "agent_trace": record["search_events"], "fixed_usage": fixed_usage, "agent_usage": agent_usage, "tool_calls": tool_calls, "fixed_latency_s": round(record["fixed_latency"], 3), "agent_latency_s": round(record["agent_latency"], 3)})
assert all(record["agent_result"]["messages"][-1].text for record in records)

{'qid': 'C01', 'fixed_answer': '전자책은 **7일**, 일반 도서는 **14일**입니다.  \n예약자가 없으면 일반 도서는 **한 번 연장**할 수 있습니다.', 'agent_answer': '일반 도서는 **14일** 동안 대출할 수 있습니다.  \n예약자가 없으면 **1회 연장**도 가능합니다.', 'fixed_doc_ids': ['LIB-004', 'LIB-001'], 'agent_trace': [{'query': '책을 빌릴 수 있는 기간 며칠 대출 기간 도서 대출', 'doc_ids': ['LIB-004', 'LIB-001']}], 'fixed_usage': {'input_tokens': 95, 'output_tokens': 37, 'model_calls': 1}, 'agent_usage': {'input_tokens': 460, 'output_tokens': 66, 'model_calls': 2}, 'tool_calls': 1, 'fixed_latency_s': 1.087, 'agent_latency_s': 2.007}
{'qid': 'C05', 'fixed_answer': '토요일 자료실은 **오후 5시**에 문을 닫습니다.', 'agent_answer': '토요일 자료실은 **오후 5시**에 폐관합니다.', 'fixed_doc_ids': ['FAC-003', 'FAC-004'], 'agent_trace': [{'query': '토요일 자료실 폐관 시각', 'doc_ids': ['FAC-003', 'FAC-004']}], 'fixed_usage': {'input_tokens': 98, 'output_tokens': 22, 'model_calls': 1}, 'agent_usage': {'input_tokens': 448, 'output_tokens': 44, 'model_calls': 2}, 'tool_calls': 1, 'fixed_latency_s': 1.875, 'agent_latency_s': 1.538}
{'qid'

## 원문 기준 LLM 평가

Ch6에서 배운 `model.with_structured_output(Grade, method="json_schema", strict=True)`는 `Grade` 형식을 적용한 새 호출 객체를 준비하며, 이 줄만으로 모델을 호출하지 않습니다. 뒤의 `grader.invoke(...)`가 원문과 답을 실제로 보내고 `Grade` 객체를 반환합니다. `supported: bool`은 참·거짓 형식을 검증하지만 판정 자체가 옳다는 보장은 아니므로 `explanation`과 원문을 함께 읽습니다. Ch21 02의 `create_agent(..., response_format=...)`이 에이전트 최종 상태의 `structured_response`를 만드는 방식과 달리, 여기서는 모델 호출의 직접 반환값이 `Grade`입니다.


In [10]:
# gold_doc_id의 원문과 각 답을 구조화 judge가 함께 읽어 사실 지지 여부와 이유를 반환합니다.
from pydantic import BaseModel
class Grade(BaseModel):
    supported: bool
    explanation: str
grader = model.with_structured_output(Grade, method="json_schema", strict=True)
for record in records:
    gold = next(item for item in documents if item["doc_id"] == record["case"]["gold_doc_id"])
    record["fixed_grade"] = grader.invoke(f"원문과 답의 사실이 맞는지 판정하세요. 원문: {gold['text']} 답: {record['fixed'].text}")
    answer = record["agent_result"]["messages"][-1].text
    record["agent_grade"] = grader.invoke(f"원문과 답의 사실이 맞는지 판정하세요. 원문: {gold['text']} 답: {answer}")
    print(record["case"]["qid"], "fixed:", record["fixed_grade"].model_dump(), "agent:", record["agent_grade"].model_dump())
assert all(isinstance(record["fixed_grade"].supported, bool) and isinstance(record["agent_grade"].supported, bool) for record in records)

C01 fixed: {'supported': True, 'explanation': '답은 원문의 내용과 일치합니다. 원문에 일반 도서는 14일 대출 가능하고, 예약자가 없으면 한 번 연장할 수 있다고 되어 있으며, 답도 이를 그대로 포함합니다. 전자책 7일 내용은 원문에 없지만, 원문과 모순되지는 않습니다.'} agent: {'supported': True, 'explanation': '답의 내용은 원문의 사실과 일치합니다. 원문에서 일반 도서는 14일 대출 가능하고, 예약자가 없으면 한 번 연장할 수 있다고 했으며, 답도 이를 같은 의미로 표현했습니다.'}


C05 fixed: {'supported': True, 'explanation': '원문에는 ‘토요일은 오후 5시에 문을 닫습니다’라고 되어 있고, 답도 ‘토요일 자료실은 오후 5시에 문을 닫습니다’라고 동일하게 말하고 있으므로 사실이 일치합니다.'} agent: {'supported': True, 'explanation': "원문에는 토요일에 자료실이 오후 5시에 문을 닫는다고 되어 있습니다. 답의 내용인 '토요일 자료실은 오후 5시에 폐관합니다.'는 원문과 일치합니다."}


C07 fixed: {'supported': True, 'explanation': '원문에는 ‘휴학생은 전자자료를 원격으로 이용할 수 없습니다’라고 명시되어 있습니다. 답도 동일한 사실을 그대로 말하고 있어 원문과 일치합니다.'} agent: {'supported': True, 'explanation': '답의 내용이 원문과 일치합니다. 원문에서 휴학생은 전자자료를 원격으로 이용할 수 없고, 방문 열람과 일반 도서 대출은 가능하다고 했으며, 답도 이를 그대로 반영하고 있습니다.'}


## 정확도·토큰·지연 집계


In [11]:
# 세 질문의 supported 비율과 두 경로의 전체 토큰·평균 지연을 집계합니다.
from statistics import mean
summary = {"fixed_accuracy": mean(record["fixed_grade"].supported for record in records), "agent_accuracy": mean(record["agent_grade"].supported for record in records), "fixed_mean_s": round(mean(record["fixed_latency"] for record in records), 3), "agent_mean_s": round(mean(record["agent_latency"] for record in records), 3), "fixed_tokens": sum(usage_total([record["fixed"]])["input_tokens"] + usage_total([record["fixed"]])["output_tokens"] for record in records), "agent_tokens": sum(usage_total(record["agent_result"]["messages"])["input_tokens"] + usage_total(record["agent_result"]["messages"])["output_tokens"] for record in records)}
print(summary)
assert summary["fixed_accuracy"] >= 0 and summary["agent_accuracy"] >= 0

{'fixed_accuracy': 1, 'agent_accuracy': 1, 'fixed_mean_s': 1.226, 'agent_mean_s': 1.879, 'fixed_tokens': 374, 'agent_tokens': 1559}


## Ch29가 읽을 실제 비교 기록 저장

질문이 항상 문서 검색을 요구하면 고정 RAG가 적은 분기와 일정한 호출 수를 제공합니다. 계산기·데이터베이스·검색 등 여러 도구 중 하나를 골라야 하거나 검색 결과를 보고 질의를 다시 작성해야 하면 에이전트 경로의 선택 능력이 의미가 있습니다. 같은 정답률이 나와도 실행 과정과 지연은 다를 수 있습니다.

세 질문 결과는 이 데이터와 이번 실행 조건의 관찰입니다. 전체 Agentic RAG가 항상 더 정확하거나 느리다고 일반화할 수 없습니다. 정확도, 검색 성공, 모델 호출 수, 입력 토큰, 지연을 각각 읽고, Ch29 평가에서는 저장한 원문 기록을 다시 사용합니다.


In [12]:
# Ch29가 같은 실행을 재현할 수 있도록 비밀값 없는 답·trace·사용량을 JSON으로 저장합니다.
public_rows = []
for record in records:
    agent_messages = record["agent_result"]["messages"]
    public_rows.append({"qid": record["case"]["qid"], "gold_doc_id": record["case"]["gold_doc_id"], "fixed_answer": record["fixed"].text, "agent_answer": agent_messages[-1].text, "fixed_doc_ids": [item["doc_id"] for item in record["fixed_hits"]], "agent_trace": record["search_events"], "fixed_usage": usage_total([record["fixed"]]), "agent_usage": usage_total(agent_messages), "tool_calls": sum(len(getattr(message, "tool_calls", [])) for message in agent_messages), "fixed_latency_s": round(record["fixed_latency"], 3), "agent_latency_s": round(record["agent_latency"], 3), "fixed_supported": record["fixed_grade"].supported, "agent_supported": record["agent_grade"].supported})
output_path = ROOT / "labs/part6/ch21-create-agent/data/rag-comparison.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(public_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", output_path.relative_to(ROOT), "rows:", len(public_rows))
assert output_path.is_file() and len(public_rows) == 3

saved: labs/part6/ch21-create-agent/data/rag-comparison.json rows: 3
